[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/04_reporting/D2_dashboard_data_export.ipynb)

# D2: Dashboard Data Export

---

## Learning Objectives

By the end of this notebook, you will be able to:
1. **Create SQLite databases** from CSV data
2. **Deploy data** to Datasette for web access
3. **Write SQL views** for different use cases
4. **Configure Datasette metadata** for rich querying

## Why This Matters

A CSV on your laptop is useful. A web-accessible database is powerful:
- Anyone can query the data without programming
- Maps and dashboards update automatically
- APIs enable integration with other systems
- Transparency builds public trust

## The Datasette Approach

Datasette turns SQLite databases into instant APIs:
```
SQLite file → Upload to server → Instant JSON API + Web UI
```

No backend code required. Just data.

---

## Overview

Export comprehensive housing data (203 projects) for web dashboards.

**Inputs:**
- `outputs/housing_projects_comprehensive.csv` (~203 projects)
- `outputs/gellerman_news_links.csv` (news URLs)

**Outputs:**
- `datasette-deploy/berkeley_housing_map.db` (SQLite database)
- `datasette-deploy/metadata.json` (Datasette configuration)
- Dashboard JSON files

---

## 1. Setup

In [1]:
# ============================================================================
# COLAB ENVIRONMENT SETUP
# ============================================================================

import os
import sys
from pathlib import Path

print('Setting up environment...')
print('='*70)

try:
    import google.colab
    IN_COLAB = True
    print('Running in Google Colab')
except ImportError:
    IN_COLAB = False
    print('Running locally')

if IN_COLAB:
    repo_path = Path('/content/berkeley-housing-analysis')
    
    if not repo_path.exists():
        print('\nCloning repository...')
        !git clone https://github.com/blockXblock/berkeley-housing-analysis.git
    
    os.chdir(repo_path)
    if str(repo_path) not in sys.path:
        sys.path.insert(0, str(repo_path))
    
    ROOT = repo_path
else:
    def find_project_root():
        current = Path.cwd()
        for path in [current] + list(current.parents):
            if (path / '00_config').exists() or (path / 'data').exists():
                return path
        return current
    
    ROOT = find_project_root()
    os.chdir(ROOT)

# Create directories
(ROOT / 'datasette-deploy').mkdir(exist_ok=True)
(ROOT / 'outputs').mkdir(exist_ok=True)

print(f'\nWorking directory: {os.getcwd()}')
print('='*70)

Setting up environment...
Running locally

Working directory: /Users/johngage/berkeley-data


In [2]:
# ============================================================================
# IMPORTS
# ============================================================================

import pandas as pd
import sqlite3
import json
from datetime import datetime
import time

print('Imports loaded')

Imports loaded


In [3]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

_cell_start_time = None

def timer_start(label=""):
    global _cell_start_time
    _cell_start_time = time.time()
    now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    if label:
        print(f'{label}')
    print(f'Started: {now}')
    print('='*70)

def timer_end():
    global _cell_start_time
    duration = time.time() - _cell_start_time
    print('='*70)
    print(f'Duration: {duration:.2f} seconds')

print('Utilities loaded')

Utilities loaded


## 2. Load Data

Load the comprehensive dataset (203 projects) which includes:
- 115 official permit projects
- ~88 media-reported projects from Gellerman's map

In [4]:
# ============================================================================
# LOAD COMPREHENSIVE DATASET
# ============================================================================

timer_start('LOADING DATA')

# Try comprehensive first, fall back to FINAL
comprehensive_path = ROOT / 'outputs/housing_projects_comprehensive.csv'
final_path = ROOT / 'data/processed/housing_projects_FINAL.csv'

if comprehensive_path.exists():
    df = pd.read_csv(comprehensive_path)
    data_source = 'comprehensive'
    print(f'Loaded comprehensive dataset: {comprehensive_path}')
elif final_path.exists():
    df = pd.read_csv(final_path)
    data_source = 'final'
    print(f'Loaded final dataset: {final_path}')
    print('Note: Run A6 and A7 notebooks to generate comprehensive dataset')
else:
    print('ERROR: No housing data found!')
    df = pd.DataFrame()

if len(df) > 0:
    print(f'\nLoaded {len(df)} projects')
    print(f'Columns: {list(df.columns)}')
    
    # Key stats
    if 'has_official_permit' in df.columns:
        official = df['has_official_permit'].sum()
        media_only = (~df['has_official_permit']).sum()
        print(f'\nBreakdown:')
        print(f'  Official permits: {official}')
        print(f'  Media-reported: {media_only}')
    
    # Coordinates
    has_coords = df['latitude'].notna().sum() if 'latitude' in df.columns else 0
    print(f'\nWith coordinates: {has_coords} ({100*has_coords/len(df):.1f}%)')

timer_end()

LOADING DATA
Started: 2026-02-26 20:02:15
Loaded comprehensive dataset: /Users/johngage/berkeley-data/outputs/housing_projects_comprehensive.csv

Loaded 156 projects
Columns: ['project_id', 'address_display', 'address_normalized', 'latitude', 'longitude', 'data_source', 'net_units', 'status', 'has_official_permit', 'has_media_coverage', 'news_urls', 'primary_news_source', 'gellerman_original_name', 'gellerman_match_score']

Breakdown:
  Official permits: 115
  Media-reported: 41

With coordinates: 127 (81.4%)
Duration: 0.01 seconds


In [5]:
# ============================================================================
# LOAD NEWS LINKS
# ============================================================================

timer_start('LOADING NEWS LINKS')

news_path = ROOT / 'outputs/gellerman_news_links.csv'

if news_path.exists():
    df_news = pd.read_csv(news_path)
    print(f'Loaded {len(df_news)} news URLs')
    
    # Filter to actual news sources
    news_sources = ['Berkeleyside', 'SFYimby', 'SF Chronicle', 'Daily Cal', 'SFGate', 'Mercury News', 'East Bay Times']
    df_news_only = df_news[df_news['source_category'].isin(news_sources)].copy()
    print(f'News articles (excluding permits/maps): {len(df_news_only)}')
    
    # Source breakdown
    print('\nBy source:')
    for source, count in df_news_only['source_category'].value_counts().items():
        print(f'  {source}: {count}')
else:
    print('News links not found - run A6 notebook first')
    df_news = pd.DataFrame()
    df_news_only = pd.DataFrame()

timer_end()

LOADING NEWS LINKS
Started: 2026-02-26 20:02:15
Loaded 2024 news URLs
News articles (excluding permits/maps): 383

By source:
  SFYimby: 267
  Berkeleyside: 96
  Daily Cal: 11
  SFGate: 5
  Mercury News: 2
  SF Chronicle: 1
  East Bay Times: 1
Duration: 0.02 seconds


## 3. Create SQLite Database

In [6]:
# ============================================================================
# CREATE SQLITE DATABASE
# ============================================================================

timer_start('CREATING SQLITE DATABASE')

db_path = ROOT / 'datasette-deploy/berkeley_housing_map.db'

# Remove existing database
if db_path.exists():
    db_path.unlink()
    print(f'Removed existing database')

# Create new database
conn = sqlite3.connect(db_path)

print(f'Creating database: {db_path}')

# ----------------------------------------
# PROJECTS TABLE
# ----------------------------------------
if len(df) > 0:
    # Prepare projects data
    df_projects = df.copy()
    
    # Ensure ID column
    if 'id' not in df_projects.columns and 'project_id' in df_projects.columns:
        df_projects['id'] = df_projects['project_id']
    elif 'id' not in df_projects.columns:
        df_projects['id'] = range(1, len(df_projects) + 1)
    
    # Add computed columns
    df_projects['is_completed'] = df_projects['status'].str.contains(
        'Completed|Certificate of Occupancy|CO Issued', 
        case=False, 
        na=False
    ).astype(int)
    
    # Write to database
    df_projects.to_sql('projects', conn, if_exists='replace', index=False)
    print(f'\nCreated projects table: {len(df_projects)} rows')

# ----------------------------------------
# NEWS COVERAGE TABLE
# ----------------------------------------
if len(df_news_only) > 0 and len(df) > 0:
    print('\nCreating news_coverage table...')
    
    # Prepare news data
    news_records = []
    
    for idx, news_row in df_news_only.iterrows():
        # Find matching project by address
        addr = news_row.get('address_normalized')
        project_id = None
        
        if addr and 'address_normalized' in df_projects.columns:
            match = df_projects[df_projects['address_normalized'] == addr]
            if len(match) > 0:
                project_id = match.iloc[0]['id']
        
        news_records.append({
            'id': idx + 1,
            'project_id': project_id,
            'url': news_row.get('url'),
            'source': news_row.get('source_category'),
            'project_name': news_row.get('project_name'),
            'date_added': datetime.now().strftime('%Y-%m-%d')
        })
    
    df_news_db = pd.DataFrame(news_records)
    df_news_db.to_sql('news_coverage', conn, if_exists='replace', index=False)
    
    linked = df_news_db['project_id'].notna().sum()
    print(f'Created news_coverage table: {len(df_news_db)} rows ({linked} linked to projects)')

# ----------------------------------------
# CREATE INDEXES
# ----------------------------------------
print('\nCreating indexes...')

cursor = conn.cursor()

# Projects indexes
cursor.execute('CREATE INDEX IF NOT EXISTS idx_projects_status ON projects(status)')
cursor.execute('CREATE INDEX IF NOT EXISTS idx_projects_data_source ON projects(data_source)') 

# News indexes
if len(df_news_only) > 0:
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_news_project_id ON news_coverage(project_id)')
    cursor.execute('CREATE INDEX IF NOT EXISTS idx_news_source ON news_coverage(source)')

conn.commit()
print('Indexes created')

# ----------------------------------------
# VERIFY
# ----------------------------------------
print('\nDatabase tables:')
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
for row in cursor.fetchall():
    cursor.execute(f"SELECT COUNT(*) FROM {row[0]}")
    count = cursor.fetchone()[0]
    print(f'  {row[0]}: {count} rows')

conn.close()

# File size
db_size = db_path.stat().st_size / 1024
print(f'\nDatabase size: {db_size:.1f} KB')

timer_end()

CREATING SQLITE DATABASE
Started: 2026-02-26 20:02:15
Removed existing database
Creating database: /Users/johngage/berkeley-data/datasette-deploy/berkeley_housing_map.db

Created projects table: 156 rows

Creating news_coverage table...
Created news_coverage table: 383 rows (82 linked to projects)

Creating indexes...
Indexes created

Database tables:
  projects: 156 rows
  news_coverage: 383 rows

Database size: 132.0 KB
Duration: 0.07 seconds


## 4. Update Datasette Metadata

In [7]:
# ============================================================================
# CREATE/UPDATE METADATA.JSON
# ============================================================================

timer_start('UPDATING METADATA')

project_count = len(df) if len(df) > 0 else 0
official_count = df['has_official_permit'].sum() if 'has_official_permit' in df.columns else project_count
media_count = project_count - official_count

# Calculate total units
total_units = int(df['net_units'].sum()) if 'net_units' in df.columns else 0

metadata = {
    "title": "Berkeley Housing Pipeline Database",
    "description": f"Track {project_count} housing development projects in Berkeley, CA ({official_count} permitted, {media_count} proposed/reported).",
    "source": "City of Berkeley Planning & Building Departments, Eric Gellerman's Berkeley Development Map",
    "source_url": "https://github.com/blockXblock/berkeley-housing-analysis",
    "license": "Public Domain (CC0)",
    "databases": {
        "berkeley_housing_map": {
            "title": "Berkeley Housing Pipeline",
            "description": f"Housing projects with timeline tracking - {total_units:,} total units",
            "tables": {
                "projects": {
                    "title": f"All Projects ({project_count})",
                    "description": f"Housing projects from permits and media - {total_units:,} total units",
                    "sort_desc": "net_units",
                    "facets": ["status", "data_source", "has_official_permit", "has_media_coverage"],
                    "plugins": {
                        "datasette-cluster-map": {
                            "latitude_column": "latitude",
                            "longitude_column": "longitude"
                        }
                    }
                },
                "news_coverage": {
                    "title": "News Coverage",
                    "description": "Media articles about housing projects",
                    "sort_desc": "date_added",
                    "facets": ["source"]
                }
            },
            "queries": {
                "projects_by_media_coverage": {
                    "title": "Projects by Media Coverage",
                    "description": "Projects ranked by number of news articles",
                    "sql": "SELECT p.address_display, p.data_source, p.net_units, COUNT(n.url) as article_count, GROUP_CONCAT(DISTINCT n.source) as sources FROM projects p LEFT JOIN news_coverage n ON p.id = n.project_id GROUP BY p.id ORDER BY article_count DESC"
                },
                "proposed_vs_permitted": {
                    "title": "Proposed vs Permitted",
                    "description": "Compare projects by data source",
                    "sql": "SELECT data_source, COUNT(*) as count, SUM(net_units) as total_units FROM projects GROUP BY data_source"
                },
                "news_by_source": {
                    "title": "News Articles by Source",
                    "description": "Count of articles by news outlet",
                    "sql": "SELECT source, COUNT(*) as count FROM news_coverage GROUP BY source ORDER BY count DESC"
                },
                "projects_without_permits": {
                    "title": "Media-Reported Projects (No Permit)",
                    "description": "Projects that appear in news but not in permit system",
                    "sql": "SELECT address_display, latitude, longitude, primary_news_source FROM projects WHERE has_official_permit = 0 ORDER BY address_display"
                }
            }
        }
    },
    "plugins": {
        "datasette-cluster-map": {
            "latitude_column": "latitude",
            "longitude_column": "longitude"
        }
    }
}

# Save metadata
metadata_path = ROOT / 'datasette-deploy/metadata.json'
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved: {metadata_path}')
print(f'\nMetadata summary:')
print(f'  Title: {metadata["title"]}')
print(f'  Projects: {project_count}')
print(f'  Tables: {list(metadata["databases"]["berkeley_housing_map"]["tables"].keys())}')
print(f'  Queries: {list(metadata["databases"]["berkeley_housing_map"]["queries"].keys())}')

timer_end()

UPDATING METADATA
Started: 2026-02-26 20:02:15
Saved: /Users/johngage/berkeley-data/datasette-deploy/metadata.json

Metadata summary:
  Title: Berkeley Housing Pipeline Database
  Projects: 156
  Tables: ['projects', 'news_coverage']
  Queries: ['projects_by_media_coverage', 'proposed_vs_permitted', 'news_by_source', 'projects_without_permits']
Duration: 0.00 seconds


## 5. Generate Dashboard JSON Files

In [8]:
# ============================================================================
# GENERATE MAP DATA (GeoJSON)
# ============================================================================

timer_start('GENERATING MAP DATA')

if len(df) > 0 and 'latitude' in df.columns:
    # Filter to projects with coordinates
    df_map = df[df['latitude'].notna() & df['longitude'].notna()].copy()
    
    # Create GeoJSON features
    features = []
    for _, row in df_map.iterrows():
        feature = {
            "type": "Feature",
            "geometry": {
                "type": "Point",
                "coordinates": [row['longitude'], row['latitude']]
            },
            "properties": {
                "address": row.get('address_display', ''),
                "units": int(row['net_units']) if pd.notna(row.get('net_units')) else None,
                "status": row.get('status', ''),
                "data_source": row.get('data_source', 'unknown'),
                "has_permit": bool(row.get('has_official_permit', True)),
                "has_news": bool(row.get('has_media_coverage', False))
            }
        }
        features.append(feature)
    
    geojson = {
        "type": "FeatureCollection",
        "features": features
    }
    
    map_path = ROOT / 'outputs/projects_map.json'
    with open(map_path, 'w') as f:
        json.dump(geojson, f)
    
    print(f'Saved: {map_path}')
    print(f'  Features: {len(features)}')
else:
    print('No coordinate data available')

timer_end()

GENERATING MAP DATA
Started: 2026-02-26 20:02:15
Saved: /Users/johngage/berkeley-data/outputs/projects_map.json
  Features: 127
Duration: 0.01 seconds


In [9]:
# ============================================================================
# GENERATE SUMMARY STATISTICS
# ============================================================================

timer_start('GENERATING SUMMARY STATISTICS')

if len(df) > 0:
    summary = {
        "generated": datetime.now().isoformat(),
        "total_projects": len(df),
        "total_units": int(df['net_units'].sum()) if 'net_units' in df.columns else 0,
        "by_data_source": {},
        "by_status": {},
        "with_coordinates": int(df['latitude'].notna().sum()) if 'latitude' in df.columns else 0,
        "with_news_coverage": int(df['has_media_coverage'].sum()) if 'has_media_coverage' in df.columns else 0
    }
    
    # By data source
    if 'data_source' in df.columns:
        for source, group in df.groupby('data_source'):
            summary['by_data_source'][source] = {
                'count': len(group),
                'units': int(group['net_units'].sum()) if 'net_units' in group.columns else 0
            }
    
    # By status
    if 'status' in df.columns:
        for status, count in df['status'].value_counts().head(10).items():
            summary['by_status'][status] = int(count)
    
    summary_path = ROOT / 'outputs/summary.json'
    with open(summary_path, 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f'Saved: {summary_path}')
    print(json.dumps(summary, indent=2))

timer_end()

GENERATING SUMMARY STATISTICS
Started: 2026-02-26 20:02:15
Saved: /Users/johngage/berkeley-data/outputs/summary.json
{
  "generated": "2026-02-26T20:02:15.763545",
  "total_projects": 156,
  "total_units": 5470,
  "by_data_source": {
    "both": {
      "count": 50,
      "units": 4167
    },
    "media_reported": {
      "count": 41,
      "units": 0
    },
    "official_permit": {
      "count": 65,
      "units": 1303
    }
  },
  "by_status": {
    "Reported in media": 41,
    "Under Review": 40,
    "Incomplete Pending Applicant": 19,
    "In Review": 17,
    "Corrections Pending Applicant": 17,
    "Pending Final Action": 12,
    "Approved": 3,
    "Pending": 3,
    "Resubmittal Pending Staff": 2,
    "Resubmittal Pending Review": 1
  },
  "with_coordinates": 127,
  "with_news_coverage": 88
}
Duration: 0.00 seconds


## 6. Verify Output Files

In [10]:
# ============================================================================
# VERIFY OUTPUT FILES
# ============================================================================

print('OUTPUT FILES')
print('='*70)

files_to_check = [
    ('datasette-deploy/berkeley_housing_map.db', 'SQLite Database'),
    ('datasette-deploy/metadata.json', 'Datasette Config'),
    ('outputs/projects_map.json', 'GeoJSON Map Data'),
    ('outputs/summary.json', 'Summary Statistics'),
]

for filepath, desc in files_to_check:
    full_path = ROOT / filepath
    if full_path.exists():
        size = full_path.stat().st_size / 1024
        print(f'[OK] {filepath:45} {size:>8.1f} KB  - {desc}')
    else:
        print(f'[--] {filepath:45} {"N/A":>8}     - {desc}')

print('\n' + '='*70)

OUTPUT FILES
[OK] datasette-deploy/berkeley_housing_map.db         132.0 KB  - SQLite Database
[OK] datasette-deploy/metadata.json                     2.7 KB  - Datasette Config
[OK] outputs/projects_map.json                         33.2 KB  - GeoJSON Map Data
[OK] outputs/summary.json                               0.7 KB  - Summary Statistics



In [11]:
# ============================================================================
# TEST DATABASE QUERIES
# ============================================================================

print('TESTING DATABASE QUERIES')
print('='*70)

db_path = ROOT / 'datasette-deploy/berkeley_housing_map.db'

if db_path.exists():
    conn = sqlite3.connect(db_path)
    
    # Test 1: Projects by data source
    print('\n1. Projects by Data Source:')
    query1 = "SELECT data_source, COUNT(*) as count, SUM(net_units) as units FROM projects GROUP BY data_source"
    result1 = pd.read_sql(query1, conn)
    display(result1)
    
    # Test 2: News coverage stats
    print('\n2. News Coverage by Source:')
    try:
        query2 = "SELECT source, COUNT(*) as count FROM news_coverage GROUP BY source ORDER BY count DESC"
        result2 = pd.read_sql(query2, conn)
        display(result2)
    except:
        print('News coverage table not available')
    
    # Test 3: Top projects by units
    print('\n3. Top 5 Projects by Units:')
    query3 = "SELECT address_display, net_units, status, data_source FROM projects WHERE net_units IS NOT NULL ORDER BY net_units DESC LIMIT 5"
    result3 = pd.read_sql(query3, conn)
    display(result3)
    
    conn.close()
else:
    print('Database not found')

TESTING DATABASE QUERIES

1. Projects by Data Source:


,data_source,count,units
0,both,50,4167.0
1,media_reported,41,NaN
2,official_permit,65,1303.0



2. News Coverage by Source:


,source,count
0,SFYimby,267
1,Berkeleyside,96
2,Daily Cal,11
3,SFGate,5
4,Mercury News,2
5,East Bay Times,1
6,SF Chronicle,1



3. Top 5 Projects by Units:


,address_display,net_units,status,data_source
0,1750 SACRAMENTO St,739.0,Under Review,official_permit
1,2276 SHATTUCK Ave,336.0,In Review,both
2,2700 SHATTUCK Ave,276.0,In Review,both
3,1914 FIFTH St,257.0,Under Review,official_permit
4,2425 DURANT Ave,250.0,Pending Final Action,both


---

## Summary

This notebook:
- Loaded comprehensive housing data (official + media-reported)
- Created SQLite database with projects and news_coverage tables
- Updated Datasette metadata with new tables and queries
- Generated GeoJSON map data and summary statistics

**Outputs:**
- `datasette-deploy/berkeley_housing_map.db` - SQLite database
- `datasette-deploy/metadata.json` - Datasette configuration
- `outputs/projects_map.json` - GeoJSON for mapping
- `outputs/summary.json` - Summary statistics

**Deployment:**
To deploy to Fly.io:
```bash
cd datasette-deploy
fly deploy
```

**Next:** Run `D3_alerts_monitoring.ipynb` for project status alerts.